# 3-тарау. nn.Module және оқыту циклі

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 3.1. Қабаттар — модульдер

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
layer = nn.Linear(in_features=3, out_features=2)
print(layer.weight.shape, layer.bias.shape)
x = torch.randn(4, 3)          # 4 мысал, әрқайсысында 3 белгі
print(layer(x).shape)
n = sum(p.numel() for p in layer.parameters())
print("Параметрлер саны:", n)

## 3.2. nn.Sequential: қабаттарды тізбектеу

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
)
print(model)
print("Параметрлер:", sum(p.numel() for p in model.parameters()))

## 3.3. Өз модуліңізді жазу

In [ ]:
class MoonNet(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.layer1 = nn.Linear(2, hidden)
        self.layer2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 2)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        return self.out(x)          # логиттер: softmax-сыз

net = MoonNet()
for name, p in net.named_parameters():
    print(f"{name:14s} {tuple(p.shape)}")

## 3.4. Шығын функциялары

In [ ]:
logits = torch.tensor([[2.0, 0.5],
                       [0.2, 1.5]])    # 2 мысал, 2 класс
target = torch.tensor([0, 1])          # дұрыс кластар
loss_fn = nn.CrossEntropyLoss()
print(loss_fn(logits, target))
probs = torch.softmax(logits, dim=1)
print(probs)
print(-torch.log(probs[[0, 1], target]).mean())   # дәл сол сан

## 3.5. Оптимизаторлар

## 3.6. Dataset және DataLoader

In [ ]:
from sklearn.datasets import make_moons
from torch.utils.data import TensorDataset, DataLoader

X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
train_ds = TensorDataset(X[:800], y[:800])
test_X, test_y = X[800:], y[800:]
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

xb, yb = next(iter(train_dl))
print(len(train_ds), len(train_dl))
print(xb.shape, yb.shape)

In [ ]:
from torch.utils.data import Dataset

class SquaresDataset(Dataset):
    def __init__(self, n):
        self.x = torch.arange(n, dtype=torch.float32)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.x[i] ** 2   # (кіріс, жауап)

ds = SquaresDataset(5)
print(len(ds), ds[3])

## 3.7. Оқыту циклі

In [ ]:
torch.manual_seed(0)
model = MoonNet()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

def accuracy(model, X, y):
    model.eval()                         # тексеру режимі
    with torch.no_grad():
        pred = model(X).argmax(dim=1)
    model.train()                        # оқыту режиміне қайту
    return (pred == y).float().mean().item()

for epoch in range(1, 31):
    for xb, yb in train_dl:
        logits = model(xb)               # 1) тура есептеу
        loss = loss_fn(logits, yb)       # 2) шығын
        opt.zero_grad()                  # 3) градиенттерді нөлдеу
        loss.backward()                  # 4) кері таралу
        opt.step()                       # 5) параметрлерді жаңарту
    if epoch % 10 == 0:
        acc = accuracy(model, test_X, test_y)
        print(f"{epoch:2d}-дәуір: шығын={loss.item():.3f}, "
              f"тест дәлдігі={acc:.3f}")

## 3.8. Оқыту және тексеру режимдері

In [ ]:
drop = nn.Dropout(p=0.5)
x = torch.ones(8)
torch.manual_seed(1)
print(drop(x))     # оқыту: кездейсоқ бөлігі 0, қалғаны ×2
drop.eval()
print(drop(x))     # тексеру: өзгеріссіз

## 3.9. Модельді сақтау

In [ ]:
torch.save(model.state_dict(), "moons.pt")

new_model = MoonNet()
new_model.load_state_dict(torch.load("moons.pt"))
print(f"{accuracy(new_model, test_X, test_y):.3f}")